# The conviction portfolio

Index-like managers hold everything, so their top positions say little.
Concentrated managers are making a statement. Rank the big holders of
one stock by portfolio concentration, keep the most concentrated books,
and aggregate their highest-conviction positions.

Needs `pandas` and `matplotlib`, and a free API key in
`THREESPREAD_API_KEY` ([signup](https://3spread.com/auth/signup)).

In [1]:
import itertools

import pandas as pd

from py3spread import Client

client = Client(max_retries=8)  # long pull, ride through the per-minute rate limit

holders = client.institutional_holdings.holdings(
    cusip="037833100", period="2026-03-31", min_value=1_000_000_000,
    sort="value", order="desc", limit=30)["data"]
managers = {}
for h in holders:
    managers.setdefault(h["filing_manager_cik"], h["filing_manager_name"])
managers = dict(list(managers.items())[:11])
managers["1067983"] = "BERKSHIRE HATHAWAY INC"  # the reference conviction book


def top_weights(cik, top_n=5):
    latest = client.institutional_holdings.list(filing_manager_cik=cik, limit=1)["data"][0]
    rows = itertools.islice(client.institutional_holdings.iter_holdings(
        filing_id=latest["filing_id"], sort="value", order="desc", limit=500), 1500)
    df = pd.DataFrame(rows)
    df["value_usd"] = pd.to_numeric(df["value_usd"])
    w = (df.groupby("name_of_issuer")["value_usd"].sum().sort_values(ascending=False))
    w = w / w.sum()
    return w.head(top_n), w.head(10).sum()


books = {}
for cik, name in managers.items():
    top5, top10_share = top_weights(cik)
    books[name] = {"top5": top5, "concentration": top10_share}

conc = pd.Series({n: b["concentration"] for n, b in books.items()}).sort_values(ascending=False)
print("top-10 concentration by manager:")
print((conc * 100).round(1).to_string())

top-10 concentration by manager:
BERKSHIRE HATHAWAY INC                      91.1
BlackRock, Inc.                             40.3
FMR LLC                                     39.8
MORGAN STANLEY                              38.4
NORTHERN TRUST CORP                         37.1
PRICE T ROWE ASSOCIATES INC /MD/            35.9
VANGUARD CAPITAL MANAGEMENT LLC             34.6
GEODE CAPITAL MANAGEMENT, LLC               32.8
STATE STREET CORP                           30.3
CHARLES SCHWAB INVESTMENT MANAGEMENT INC    29.7
JPMORGAN CHASE & CO                         29.3
VANGUARD PORTFOLIO MANAGEMENT LLC           23.1


## Aggregate the concentrated books

Keep the most concentrated books: those with more than half in their
top 10 when any qualify, otherwise the three most concentrated (big-cap
holder lists skew to diversified custodians). Then sum their top-5
weights per issuer:

In [2]:
convicted = [n for n, c in conc.items() if c > 0.5]
if len(convicted) < 3:
    # big-cap holder lists skew to diversified custodians, so fall back
    # to the three most concentrated books
    convicted = list(conc.head(3).index)
print("conviction managers:", ", ".join(convicted))

agg = {}
for name in convicted:
    for issuer, w in books[name]["top5"].items():
        agg.setdefault(issuer, {"weight": 0.0, "held_by": 0})
        agg[issuer]["weight"] += float(w)
        agg[issuer]["held_by"] += 1

clone = (pd.DataFrame(agg).T.sort_values("weight", ascending=False).head(12))
clone["weight"] = clone["weight"] / len(convicted)
clone.style.format({"weight": "{:.1%}", "held_by": "{:.0f}"})

conviction managers: BERKSHIRE HATHAWAY INC, BlackRock, Inc., FMR LLC


,weight,held_by
APPLE INC,11.1%,3
NVIDIA CORPORATION,6.0%,2
AMERICAN EXPRESS CO,5.8%,1
COCA COLA CO,3.9%,1
ALPHABET INC,3.7%,2
BANK AMERICA CORP,3.2%,1
MICROSOFT CORP,3.1%,2
AMAZON COM INC,2.6%,2
CHEVRON CORPORATION,2.2%,1


`weight` is the average allocation across the conviction books;
`held_by` counts how many of them hold the name top-5. This composes
with the similarity matrix in `institutional_13f.ipynb`: overlap tells
you who thinks independently, concentration tells you who commits.